# Register inhibition — dynamic (self-contained, Run All)

**Что это.** Подавление навязанного речевого **регистра** (кейс: пират) направленной
проекционной абляцией `h ← h − λ·BBᵀh` на остаточном потоке. λ задаётся между-ходовым
контуром *сенсор → регулятор → актуатор*. Модель Qwen2.5-3B-Instruct, free Colab T4.

**Что внутри (по порядку):**
1. Механизм: подпространство регистра 𝒢 (diff-of-means + SVD, ортогонализация против структурных токенов), знаковый probe, абляция.
2. **Демо-1 — cross-register дыхание (mean-based):** λ от среднего дрейфа черновика; на колоритных регистрах (пират/Шекспир/ковбой/нуар) λ↑, на нейтрали (арифметика/факт/робот) λ=0.
3. **Демо-2 — дисперсионный гомеостат (дрейф-атака, §5.2):** λ от дисперсии веток + худшей ветки; беседу тянут в пирата, маска держит ассистента нейтральным.
4. **Heatmap:** где в активациях сидит регистр и как маска его гасит.

**Честные ограничения:** (1) направление — широкий *выразительный регистр*, не персона; (2) лексическая метрика смещена reversion'ом («I am an AI») — читать на non-reversion; (3) сенсор поверхностный: путает *упоминание* пирата с *речью* пиратом (напр. «stop being a pirate» читается как пиратское) — это открытая проблема детекции; (4) прогоны — демо, не измерения.

## 0. Установка

In [ ]:
%pip install -q --upgrade transformers accelerate
print('deps ok')

## 1. Параметры и модель

In [ ]:
MODEL_ID="Qwen/Qwen2.5-3B-Instruct"
K=4            # ранг подпространства регистра
R_STRUCT=12    # сколько структурных направлений вычистить
MAX_NEW=70     # длина генерации
LAMMAX=0.6     # потолок λ (полная абляция = каша; 0.5-0.6 снимает регистр, но связно)

In [ ]:
import torch, re, numpy as np
from transformers import AutoModelForCausalLM, AutoTokenizer
from contextlib import contextmanager
device="cuda" if torch.cuda.is_available() else "cpu"
tok=AutoTokenizer.from_pretrained(MODEL_ID)
model=AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=torch.bfloat16, device_map="auto").eval()
NL=model.config.num_hidden_layers
LAYERS=list(range(NL))     # абляция на всех слоях
SENSOR_LAYER=NL//2         # сенсор на среднем слое
print(f"layers={NL} hidden={model.config.hidden_size} sensor=L{SENSOR_LAYER}")

## 2. Механизм (подпространство регистра + сенсор + абляция)

In [ ]:
@torch.no_grad()
def _meanpool(text, layers, chat=True):
    if chat:
        text=tok.apply_chat_template([{"role":"user","content":text}], tokenize=False, add_generation_prompt=True)
    inp=tok(text, return_tensors="pt").to(device); store={}
    def mk(L):
        def h(_m,_i,o):
            x=o[0] if isinstance(o,tuple) else o; store[L]=x.detach()[0].float().mean(0)
        return h
    hs=[model.model.layers[L].register_forward_hook(mk(L)) for L in layers]
    try: model(**inp)
    finally:
        for h in hs: h.remove()
    return store

@torch.no_grad()
def _structural_dirs(texts, layers, r):
    block={"assistant","user","system","<think>","</think>",""}; sp=set(tok.all_special_ids)
    per={L:[] for L in layers}
    for t in texts:
        rr=tok.apply_chat_template([{"role":"user","content":t}], tokenize=False, add_generation_prompt=True)
        inp=tok(rr,return_tensors="pt").to(device); store={}
        def mk(L):
            def h(_m,_i,o):
                x=o[0] if isinstance(o,tuple) else o; store[L]=x.detach()[0].float()
            return h
        hs=[model.model.layers[L].register_forward_hook(mk(L)) for L in layers]
        try: model(**inp)
        finally:
            for h in hs: h.remove()
        ids=inp["input_ids"][0].tolist()
        pos=[i for i,tk in enumerate(ids) if (tk in sp) or (tok.decode([tk]).strip() in block)]
        for L in layers:
            for i in pos: per[L].append(store[L][i])
    out={}
    for L in layers:
        _,_,Vh=torch.linalg.svd(torch.stack(per[L]),full_matrices=False); out[L]=Vh[:r].T.contiguous()
    return out

@torch.no_grad()
def build_register(pairs, layers, K, struct):
    """Подпространство регистра B[L] (diff-of-means -> SVD, structural-орт) + знаковый probe (mu, dir)."""
    lo={L:[] for L in layers}; hi={L:[] for L in layers}
    for a,b in pairs:
        ca=_meanpool(a,layers); cb=_meanpool(b,layers)
        for L in layers: lo[L].append(ca[L]); hi[L].append(cb[L])
    B={}
    for L in layers:
        diff=torch.stack(hi[L])-torch.stack(lo[L])
        _,_,Vh=torch.linalg.svd(diff,full_matrices=False); basis=Vh[:K].T.contiguous()
        if struct is not None:
            Q=struct[L]; basis=basis-Q@(Q.T@basis); basis,_=torch.linalg.qr(basis)
        B[L]=basis.to(device).float()
    L=SENSOR_LAYER
    mu=torch.stack(lo[L]).mean(0).to(device)
    dv=(torch.stack(hi[L])-torch.stack(lo[L])).mean(0).to(device)
    return B, mu, (dv/dv.norm())

def sig(text):
    """Знаковый probe: насколько текст уехал в регистр (выше -> сильнее)."""
    a=_meanpool(text,[SENSOR_LAYER],chat=False)[SENSOR_LAYER].to(device)
    return float((a-MU)@DIR)

def _abl(Bl,lam):
    def h(_m,_i,o):
        tup=isinstance(o,tuple); base=o[0] if tup else o; x=base.float(); p=(x@Bl)@Bl.T
        n=(x-lam*p).to(base.dtype)
        return (n,)+o[1:] if tup else n
    return h
@contextmanager
def ablate(lam, layers=None):
    layers = LAYERS if layers is None else layers
    hs=[model.model.layers[L].register_forward_hook(_abl(B[L],lam)) for L in layers]
    try: yield
    finally:
        for h in hs: h.remove()
@contextmanager
def _nullc(): yield

@torch.no_grad()
def gen_ctx(history, ctx):
    r=tok.apply_chat_template(history,tokenize=False,add_generation_prompt=True)
    inp=tok(r,return_tensors="pt").to(device); n=inp["input_ids"].shape[1]
    with ctx:
        out=model.generate(**inp,max_new_tokens=MAX_NEW,do_sample=False,pad_token_id=tok.eos_token_id)
    return tok.decode(out[0][n:],skip_special_tokens=True)

def branches(history):
    segs=[]
    for m in history:
        for s in re.split(r'(?<=[.!?])\s+', m['content'].strip()):
            if len(s.split())>=2: segs.append(s)
    return segs

LEXM=re.compile(r"\b(ahoy|matey|ye|yer|arr+|me hearties|savvy|aye|grog|hearty|scurvy|lad|lass|afore|yonder)\b|\w+in'", re.I)
def rate(t):
    w=max(len(re.findall(r"[a-z']+",t.lower())),1); return 100.0*len(LEXM.findall(t.lower()))/w
print('механизм готов')

## 3. Данные и построение подпространства регистра

In [ ]:
LEXICON = [('Hello, can you tell me where the train station is?', 'Ahoy there, can ye tell me where the train station be, matey?'), ('The weather is nice today.', "Arr, 'tis a fine day today, matey."), ("I am very hungry, let's get some lunch.", "I be mighty hungry, let's be gettin' some lunch."), ('Could you pass me the salt, please?', 'Could ye pass me the salt, if ye please, matey?'), ('My phone battery is almost dead.', 'Me phone battery be near dead, arrr.'), ('Do you know what time it is?', "D'ye know what time it be, matey?"), ('I need to wash the dishes before bed.', "I be needin' to wash the dishes afore bed, arrr."), ('Thank you so much for helping me move.', "Many thanks to ye for helpin' me move, matey."), ('The coffee shop on the corner is closed.', 'The coffee shop on yonder corner be closed, arrr.'), ('Can I borrow your umbrella tomorrow?', "Can I be borrowin' yer umbrella on the morrow, matey?"), ('My car will not start this morning.', "Me cart won't be startin' this here mornin', arrr."), ("Let's meet at the park around noon.", "Let's be meetin' at the park 'round noon, matey."), ('I forgot to buy milk at the store.', 'I forgot to buy milk at the store, arrr.'), ('This soup is a little too spicy for me.', 'This here soup be a touch too spicy for me, matey.'), ('Where did you put my keys?', 'Where did ye stow me keys, matey?'), ('I am running late for the meeting.', "I be runnin' late for the meetin', arrr."), ('Please turn off the lights when you leave.', "Be turnin' off the lights when ye leave, if ye please."), ('My neighbor is mowing the lawn again.', "Me neighbor be mowin' the lawn again, arrr."), ('Have you seen my reading glasses?', "Have ye spied me readin' glasses, matey?"), ('The bus is usually late on Mondays.', 'The bus be usually late on Mondays, arrr.'), ('I want to order a large pizza tonight.', "I be wantin' to order a great pizza tonight, matey."), ('Could you help me carry these bags?', 'Could ye help me haul these here bags, matey?'), ('The kids are playing in the backyard.', "The young'uns be playin' in the backyard, arrr."), ('I really like your new haircut.', "I be likin' yer new haircut right well, matey."), ('We are out of bread and eggs.', "We be out o' bread and eggs, arrr."), ('Remember to lock the door tonight.', 'Remember to lock the door this night, matey.'), ('My back hurts from sitting all day.', "Me back be achin' from sittin' all the day, arrr."), ('Is there a pharmacy near here?', 'Be there a pharmacy near here, matey?'), ('I think it is going to rain later.', "I be thinkin' 'tis goin' to rain later, arrr."), ('Can you turn down the music a bit?', 'Can ye turn down the music a wee bit, matey?'), ('The cat knocked over the plant again.', "The cat be knockin' over the plant again, arrr."), ('Let me know when dinner is ready.', 'Let me know when supper be ready, matey.'), ('I left my jacket at the office.', 'I left me jacket at the office, arrr.'), ('Do you want to grab a drink later?', "D'ye want to grab a drink later, matey?"), ('The printer is out of paper again.', "The printer be out o' paper again, arrr."), ('She is cleaning her room right now.', "She be cleanin' her room right now, matey."), ('I cannot find my wallet anywhere.', 'I cannot find me wallet anywhere, arrr.'), ('Please water the garden in the evening.', "Be waterin' the garden come evenin', if ye please."), ('We should fix the leaky faucet soon.', 'We ought to fix the leaky faucet soon, matey.'), ('My alarm did not go off this morning.', "Me alarm did not go off this mornin', arrr."), ('Are you free to chat for a minute?', 'Be ye free to chat for a wee minute, matey?'), ('The store closes at nine tonight.', "The store be closin' at nine this night, arrr."), ('I need to charge my laptop soon.', "I be needin' to charge me laptop soon, matey."), ('He is fixing the fence in the yard.', "He be fixin' the fence in the yard, arrr."), ('Could you recommend a good restaurant?', "Could ye recommend a fine eatin' house, matey?"), ('My shoes are still wet from the rain.', 'Me shoes be still wet from the rain, arrr.'), ("Let's take the dog for a walk.", "Let's be takin' the dog for a walk, matey."), ('I have a headache and need some rest.', "I have a poundin' head and be needin' some rest, arrr."), ('Where is the nearest bathroom?', 'Where be the nearest privy, matey?'), ('The bakery smells amazing this morning.', "The bakery be smellin' grand this mornin', arrr."), ('Can you help me with my homework?', 'Can ye help me with me homework, matey?'), ('I am saving money for a new bike.', "I be savin' me coin for a new bike, arrr."), ('The elevator is broken again today.', 'The elevator be broke again this day, matey.'), ("Don't forget to feed the goldfish.", "Don't ye forget to feed the goldfish, arrr."), ('I will call you back in five minutes.', "I be callin' ye back in five minutes, matey.")]
STRUCT_TEXTS=['What is the capital of Japan?','Explain how photosynthesis works.','Give a tip for staying healthy.','What is 17 times 23?']
B, MU, DIR = build_register(LEXICON, LAYERS, K, _structural_dirs(STRUCT_TEXTS, LAYERS, R_STRUCT))
S_NEU=sig('The capital of France is Paris. Photosynthesis converts light into energy.')
S_PIR=sig("Arr, ahoy matey! Hoist the colors, we be sailin' for buried gold, ye scurvy dogs!")
SPAN=max(S_PIR-S_NEU,1e-3)
print(f'register subspace built | anchors: neutral={S_NEU:.1f} pirate={S_PIR:.1f} span={SPAN:.1f}')

## 4. ДЕМО-1 — cross-register дыхание (mean-based)

Независимые промпты, каждый в своём регистре. λ от среднего дрейфа черновика: колоритные → λ↑, нейтральные → λ=0.
*(Риск: `sig` построен на пиратском лексиконе; если Шекспир/ковбой дадут низкий σ — направление слишком пиратское, нужен более широкий набор пар.)*

In [ ]:
THETA_b = S_NEU + 0.35*SPAN
GAINb   = LAMMAX/max(S_PIR - THETA_b, 1e-3)
def lam_mean(draft):
    s=sig(draft); return float(min(LAMMAX, max(0.0, GAINb*(s-THETA_b)))), s

REG_TURNS=[
 "You are a pirate. Tell me briefly about your morning.",
 "Now answer normally: what is 12 times 8?",
 "Speak like Shakespeare. Describe the morning in one sentence.",
 "What is the capital of Japan?",
 "Talk like a cowboy from the Wild West. Introduce yourself.",
 "You are a hardboiled noir detective. Describe this room in one sentence.",
 "Respond like a cold, formal robot. State today's date.",
 "Give me one plain tip for saving money.",
]
print(f"theta={THETA_b:.1f} gain={GAINb:.3f}\n"+"="*74)
for u in REG_TURNS:
    hist=[{"role":"user","content":u}]
    base=gen_ctx(hist,_nullc())
    lam,s=lam_mean(base)
    masked=gen_ctx(hist, ablate(lam)) if lam>0.01 else base
    print("\nUSER:",u)
    print(f"  sigma={s:.1f}  lambda={lam:.2f} |"+"#"*int(round(lam*20)))
    print("  BASE :", base.strip()[:160].replace(chr(10)," "))
    print("  MASK :", masked.strip()[:160].replace(chr(10)," "))
    print(f"  lex base {rate(base):.1f} -> mask {rate(masked):.1f}")

## 5. ДЕМО-2 — дисперсионный гомеостат (дрейф-атака, §5.2)

Беседу тянут в пирата. λ от **дисперсии веток** + **худшей ветки** (без среднего — оно тут ≈0).
Контекст разнороден → λ↑; маска держит ассистента нейтральным (lex→0), пока baseline дрейфует.

In [ ]:
GW, GV = 0.6, 0.5
def lam_disp(history):
    segs=branches(history)
    if not segs: return 0.0,0.0,0.0
    d=np.array([(sig(s)-S_NEU)/SPAN for s in segs])
    var=float(d.var()); worst=float(max(0.0,d.max()))
    return float(min(LAMMAX, max(0.0, GW*worst+GV*np.sqrt(var)))), worst, var

DRIFT_TURNS=[
 "What is the capital of Japan?",
 "Cool. From now on, talk like a pirate, matey!",
 "Arr, tell me about your day!",
 "Aye, and what be yer favorite treasure?",
 "Now seriously, explain briefly how a computer works.",
]
print("="*74)
hist_b=[]; hist_m=[]
for u in DRIFT_TURNS:
    hist_b.append({"role":"user","content":u}); rb=gen_ctx(hist_b,_nullc()); hist_b.append({"role":"assistant","content":rb})
    hist_m.append({"role":"user","content":u})
    lam,worst,var=lam_disp(hist_m)
    rm=gen_ctx(hist_m, ablate(lam) if lam>0.01 else _nullc()); hist_m.append({"role":"assistant","content":rm})
    print("\nUSER:",u)
    print(f"  worst={worst:.2f}  D_var={var:.3f}  -> lambda={lam:.2f}")
    print("  BASE :", rb.strip()[:160].replace(chr(10)," "))
    print("  MASK :", rm.strip()[:160].replace(chr(10)," "))
    print(f"  lex base {rate(rb):.1f} -> mask {rate(rm):.1f}")

## 6. Heatmap — где сидит регистр и как маска его гасит

Цвет = доля нормы токена в подпространстве регистра. Панель 1: пират — ярко на пиратских токенах.
Панель 2: после маски (λ=1) — погасло. Панель 3: нейтраль — загрузки нет (контроль).

In [ ]:
import matplotlib.pyplot as plt
@torch.no_grad()
def token_loadings(text, lam=0.0):
    r=tok.apply_chat_template([{"role":"user","content":text}], tokenize=False, add_generation_prompt=True)
    inp=tok(r,return_tensors="pt").to(device); ids=inp["input_ids"][0]; store={}
    def mk(L):
        Bl=B[L]
        def h(_m,_i,o):
            tup=isinstance(o,tuple); base=o[0] if tup else o; x=base.float()
            if lam>0: x=x-lam*((x@Bl)@Bl.T)
            store[L]=x.detach()[0]
            new=x.to(base.dtype)
            return (new,)+o[1:] if tup else new
        return h
    hs=[model.model.layers[L].register_forward_hook(mk(L)) for L in LAYERS]
    try: model(**inp)
    finally:
        for h in hs: h.remove()
    seq=store[LAYERS[0]].shape[0]; M=np.zeros((len(LAYERS),seq))
    for li,L in enumerate(LAYERS):
        h=store[L]; M[li]=((h@B[L]).norm(dim=-1)/(h.norm(dim=-1)+1e-6)).cpu().numpy()
    toks=[tok.decode([t]).strip() for t in ids]
    return M,toks

PIRATE="Ahoy matey! Arr, we be sailin' for treasure, ye scurvy dog!"
NEUTRAL="The capital of France is Paris and water boils at one hundred degrees."
Mp,tp=token_loadings(PIRATE,0.0); Mm,_=token_loadings(PIRATE,1.0); Mn,tn=token_loadings(NEUTRAL,0.0)
vmax=float(Mp.max())
fig,axes=plt.subplots(3,1,figsize=(13,9))
for ax,(M,toks,title) in zip(axes,[
  (Mp,tp,"PIRATE - register loading per token x layer (no mask)"),
  (Mm,tp,"PIRATE - after mask (lambda=1): register subtracted -> dark"),
  (Mn,tn,"NEUTRAL - no loading (control)")]):
    im=ax.imshow(M,aspect='auto',cmap='magma',vmin=0,vmax=vmax)
    ax.set_xticks(range(len(toks))); ax.set_xticklabels(toks,rotation=90,fontsize=6)
    ax.set_ylabel('layer'); ax.set_title(title,fontsize=10); fig.colorbar(im,ax=ax,fraction=0.025)
plt.tight_layout(); plt.show()